# 16 - Measurement Validity Gate: Integrity Handling and Recovery Tests

**Purpose.** Validation of the Measurement Validity Gate (MVG), a lightweight preprocessing/supervisory layer placed in
front of the unchanged CRI supervisor: (i) fault-free equivalence with the reference implementation
(`sim.run_closed_loop`) on S1-S8, and (ii) focused fault-injection tests (non-finite values/dropouts, stale or delayed
packets, frozen channels, outliers), including fallback handling and recovery.

**MVG vs. NCI.** The Noise-Confidence Index (NCI) is one of the five readiness indicators and grades how trustworthy the
*estimated sensitivity* is under measurement noise. The MVG is a separate, binary check on each *delivered sample*;
it decides whether that sample may be used at all. The gate does not change NCI or any other indicator.

**Scope.** Validity gating with a requested fallback transition. This is not sensor-fault diagnosis or isolation, not
fault-tolerant control, and not a safety guarantee: under invalid data the fallback runs on the last usable (held)
measurement, and safe operation still relies on the fallback controller keeping the plant admissible. The gate does
not detect every possible sensor fault (see the undetected cases in Sec. 5).

**Code.** `cri_microgrid/mvg.py` (gate), `cri_microgrid/mvg_sim.py` (CRI closed loop with gate and fault-injection
hooks; reproduces the `'cri'` path of `sim.run_closed_loop`). `sim.py`, `cri.py`, `sensitivity.py`,
`controllers.py` and all CRI equations/parameters are unchanged.

**Outputs.** `results/r1c8_*.csv`, `results/r1c8_run_metadata.json`. Runtime about 5-6 min on a desktop CPU.


In [1]:
import sys, os, time, json, platform, warnings, dataclasses, datetime
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, scipy
from cri_microgrid.config import (PlantParams, ControllerParams, ConstraintParams, CRIParams, ProbeParams,
                                   ProfileParams, SimulationParams)
from cri_microgrid import sim, metrics, validation
from cri_microgrid import dc_calibration as dcc
from cri_microgrid import mvg, mvg_sim
from cri_microgrid.mvg_sim import Fault, FaultInjector
from cri_microgrid.sim import LOG_FIELDS
from cri_microgrid.benchmark_eval import scenario_builders, load_dc_fold_params, SCENARIO_ORDER, NOISY

RESULTS_DIR = os.path.abspath("../results")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
T_START = time.time()
pp, cp, cons, crip, probep, prof = PlantParams(), ControllerParams(), ConstraintParams(), CRIParams(), ProbeParams(), ProfileParams()
B = scenario_builders(pp, prof)
MVG_PRM = mvg.MVGParams()
SEED_REP = 1
SEEDS_EC = {s: sorted(set([1] + (list(range(20)) if s in NOISY else []))) for s in SCENARIO_ORDER}   # = Protocol A of nb 15
SEEDS_LOSO = {s: ([0, 1, 2, 3, 4] if s in NOISY else [1]) for s in SCENARIO_ORDER}                   # = notebook 13
DC_FOLD = load_dc_fold_params(os.path.join(RESULTS_DIR, "results_dc_cri_loso_folds.csv"), crip)
R_DEFAULT = mvg.MeasurementValidityGate(MVG_PRM, crip).R
meta = dict(created=datetime.datetime.now().isoformat(timespec="seconds"), python=platform.python_version(),
            numpy=np.__version__, scipy=scipy.__version__, pandas=pd.__version__, platform=platform.platform(),
            mvg=dataclasses.asdict(MVG_PRM), recovery_samples_resolved=R_DEFAULT, seeds_ec=SEEDS_EC, seeds_loso=SEEDS_LOSO)
print(json.dumps(meta, indent=1, default=str))

def run_ref(sname, seed, cri_p):
    sc = B[sname](); return sim.run_closed_loop(sc, pp, cp, cons, cri_p, probep, SimulationParams(T_end=sc.T_end, seed=seed, strategy="cri"))
def run_mvg(sname, seed, cri_p, gate_prm=None, use_gate=True, faults=None, timestamps=True, catch=False):
    sc = B[sname](); g = mvg.MeasurementValidityGate(gate_prm or MVG_PRM, cri_p) if use_gate else None
    fi = FaultInjector(faults, timestamps) if faults else None
    lg = mvg_sim.run_closed_loop_mvg(sc, pp, cp, cons, cri_p, probep, SimulationParams(T_end=sc.T_end, seed=seed),
                                     gate=g, fault=fi, catch_exceptions=catch)
    lg["_sc"], lg["_gate"] = sc, g
    return lg
def perf(lg):
    sc = lg["_sc"] if "_sc" in lg else None
    z = validation.compute_unsafe_truth(lg, cons, noise_level_pct=sc.noise_cfg.get("level_pct", 0.0), noise_window=sc.noise_window)
    zH = dcc.finite_horizon_label(lg, cons, 25); s = lg["s"]
    return dict(N_FA=metrics.false_activation_count(s, z), active_in_hazard_pct=100 * metrics.unsafe_active_fraction(s, z),
                N_FA_H=metrics.false_activation_count_horizon(s, zH), FAR=metrics.far(s, zH), FDR=metrics.fdr(s, zH),
                A_c=metrics.controller_availability(s), E_V=metrics.voltage_deviation(lg["V1"], lg["V2"], pp.Vac_nom),
                n_violation_true=int(dcc.hard_limit_violation(lg, cons).sum()), n_switch=int(np.sum(np.abs(np.diff(s)) > 0.5)),
                dt_flag_pct=100 * float(np.mean(lg["dt_flag"])))

{
 "created": "2026-11-20T00:58:56",
 "python": "3.10.9",
 "numpy": "2.2.6",
 "scipy": "1.15.3",
 "pandas": "2.3.3",
 "platform": "Windows-10-10.0.19045-SP0",
 "mvg": {
  "y_lo": [
   0.3,
   0.3,
   0.3,
   -0.2,
   -0.2,
   -5.0,
   -5.0
  ],
  "y_hi": [
   1.7,
   1.7,
   1.7,
   3.0,
   3.0,
   5.0,
   5.0
  ],
  "tau_max": 0.03,
  "frozen_L": 10,
  "frozen_tol": 0.0,
  "u_excitation_min": 1e-09,
  "zero_exempt_channels": [
   3,
   4
  ],
  "zero_tol": 1e-09,
  "channel_inputs": [
   [
    0,
    1,
    2,
    3
   ],
   [
    0,
    1,
    2,
    3
   ],
   [
    0,
    2
   ],
   [
    0,
    1
   ],
   [
    2,
    3
   ],
   [
    0,
    2
   ],
   [
    1,
    3
   ]
  ],
  "tau_fb_max": 0.2,
  "recovery_samples": null,
  "startup_requires_recovery": false
 },
 "recovery_samples_resolved": 40,
 "seeds_ec": {
  "S1": [
   1
  ],
  "S2": [
   1
  ],
  "S3": [
   1
  ],
  "S4": [
   1
  ],
  "S5": [
   1
  ],
  "S6": [
   0,
   1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,


## Gate definition (implemented in `cri_microgrid/mvg.py`)

| Check | Rule (defaults) | Rejects |
|---|---|---|
| finite | all 7 channels finite | NaN/Inf, dropped packets |
| fresh | $t_k-\tau_k\le\tau_{\max}=0.03$ s and $\tau_k>\tau_{k-1}$ (only if timestamps exist) | stale, repeated, delayed packets |
| plausible | $V_1,V_2,V_{dc}\in[0.3,1.7]$, $I_{1,2}\in[-0.2,3]$, $P_{POI},Q_{POI}\in[-5,5]$ pu | gross outliers only |
| not frozen | no channel repeats exactly for $L=10$ samples **while** an input that physically drives that channel changed ($>10^{-9}$) at each of them ($V_{1,2}$: all inputs; $V_{dc}$, $P_{POI}$: $P_1^*,P_2^*$; $Q_{POI}$: $Q_1^*,Q_2^*$; $I_i$: $P_i^*,Q_i^*$); exactly-zero currents exempt | stuck-at (non-zero) channels |

Invalid sample: no CRI update, $s_k=0$, candidate update frozen, DT flag raised. The fallback receives each channel's current
value if that channel is usable for it (finite, plausible, not frozen, packet age $\le0.2$ s) and otherwise that channel's last
usable value; an invalid value never reaches the fallback. With no usable channel the fallback runs on held data, i.e. open loop.
Recovery: CRI state reset on the first valid sample; $s_k=0$ and DT flag held for $R=\max(N,2N_a)=40$ consecutive valid samples,
so every CRI data window (sensitivity window $N$, adaptability window $2N_a$) contains post-fault data only.
At cold start the gate does **not** impose $R$ (default), so fault-free behaviour equals the reference implementation; Sec. 6 evaluates the alternative.

## 1. Reference baseline (unchanged `sim.run_closed_loop`)

In [2]:
ref = pd.read_csv(os.path.join(RESULTS_DIR, "results_strategy_comparison.csv"))
base_rows, mism = [], []
for sname in SCENARIO_ORDER:
    lg = run_ref(sname, SEED_REP, crip); lg["_sc"] = B[sname](); r = perf(lg); r.update(scenario=sname); base_rows.append(r)
    rr = ref[(ref.scenario == sname) & (ref.strategy == "cri")].iloc[0]
    if r["N_FA"] != rr.N_FA or abs(r["E_V"] - rr.E_V) > 1e-12: mism.append(sname)
base = pd.DataFrame(base_rows).set_index("scenario")
base.to_csv(os.path.join(RESULTS_DIR, "r1c8_baseline_unchanged_code.csv"))
print(base.round(4)); print("Matches results_strategy_comparison.csv (N_FA, E_V):", not mism, mism)
assert not mism

          N_FA  active_in_hazard_pct  N_FA_H     FAR     FDR     A_c     E_V  n_violation_true  n_switch  dt_flag_pct
scenario                                                                                                             
S1           0                0.0000       0  0.0000  0.0015  0.9985  0.0023                 0         1       0.3998
S2           0                0.0000       0  0.0000  0.0015  0.9985  0.0197                 0         1      46.2269
S3           9                1.1984       0  0.0000  0.4038  0.5962  0.0105                 0         5      43.6282
S4           0                0.0000       0  0.0000  0.0015  0.9985  0.0069                 0         1       1.8991
S5         499               43.8874      20  0.2410  0.5704  0.4218  0.0165                 8        68      75.7621
S6           0                0.0000       0  0.0000  0.0015  0.9985  0.0106                 0         1      75.2624
S7           5                0.4541      29  0.0258  0.

## 2. Fault-free equivalence: reference code vs. MVG runner (gate off) vs. MVG runner (gate on)

EC-CRI on the Protocol-A seed set (seed 1; seeds 0-19 for S6, S8) and DC-CRI with the stored LOSO fold parameters on
the notebook-13 seed set. Every logged field in `sim.LOG_FIELDS` (switching $s_k$, class, $\kappa_k$, CRI and all five
indicators, $\hat\sigma^2$, $\epsilon_s$, measured and true outputs, commands) is compared, including NaN patterns.

In [3]:
eq_rows = []
cases = [("EC-CRI", s, sd, crip) for s in SCENARIO_ORDER for sd in SEEDS_EC[s]] + \
        [("DC-CRI", s, sd, DC_FOLD[s]) for s in SCENARIO_ORDER for sd in SEEDS_LOSO[s]]
def maxdiff(a, b):
    d, nanmis = 0.0, 0
    for k in LOG_FIELDS:
        x, y = np.asarray(a[k], float), np.asarray(b[k], float)
        nanmis += int(np.sum(np.isnan(x) != np.isnan(y)))
        m = np.isfinite(x) & np.isfinite(y)
        if m.any(): d = max(d, float(np.max(np.abs(x[m] - y[m]))))
    return d, nanmis
for meth, sname, seed, cp_ in cases:
    a = run_ref(sname, seed, cp_); a["_sc"] = B[sname]()
    b = run_mvg(sname, seed, cp_, use_gate=False)
    c = run_mvg(sname, seed, cp_, use_gate=True)
    d_ab, n_ab = maxdiff(a, b); d_ac, n_ac = maxdiff(a, c)
    pa, pc = perf(a), perf(c)
    eq_rows.append(dict(method=meth, scenario=sname, seed=seed, maxdiff_gate_off=d_ab, nanmismatch_gate_off=n_ab,
                        maxdiff_gate_on=d_ac, nanmismatch_gate_on=n_ac, gate_invalid_samples=c["_gate"].n_invalid,
                        metrics_identical=all(pa[k] == pc[k] for k in pa),
                        **{f"ref_{k}": v for k, v in pa.items()}))
eq = pd.DataFrame(eq_rows)
eq.to_csv(os.path.join(RESULTS_DIR, "r1c8_equivalence_fault_free.csv"), index=False)
summary = eq.groupby("method").agg(runs=("seed", "size"), max_diff_gate_off=("maxdiff_gate_off", "max"),
                                   max_diff_gate_on=("maxdiff_gate_on", "max"), nan_mismatch=("nanmismatch_gate_on", "sum"),
                                   gate_invalid=("gate_invalid_samples", "sum"), metrics_identical=("metrics_identical", "all"))
print(summary)
BIT_IDENTICAL = bool((eq.maxdiff_gate_off == 0).all() and (eq.maxdiff_gate_on == 0).all()
                     and (eq.nanmismatch_gate_on == 0).all() and (eq.gate_invalid_samples == 0).all())
print("Bit-identical on every run and every logged field:", BIT_IDENTICAL, f"({time.time()-T_START:.0f} s)")

        runs  max_diff_gate_off  max_diff_gate_on  nan_mismatch  gate_invalid  metrics_identical
method                                                                                          
DC-CRI    16                0.0               0.0             0             0               True
EC-CRI    46                0.0               0.0             0             0               True
Bit-identical on every run and every logged field: True (210 s)


## 3. Margin of the fault-free runs to the plausibility envelope

In [4]:
chan = ["V1", "V2", "Vdc", "I1", "I2", "P_POI", "Q_POI"]
obs_lo = np.full(7, np.inf); obs_hi = np.full(7, -np.inf)
for meth, sname, seed, cp_ in cases:
    lg = run_ref(sname, seed, cp_)
    Y = np.c_[[lg[c] for c in chan]].T
    obs_lo = np.minimum(obs_lo, Y.min(0)); obs_hi = np.maximum(obs_hi, Y.max(0))
env = pd.DataFrame(dict(channel=chan, bound_lo=MVG_PRM.y_lo, observed_min=obs_lo, observed_max=obs_hi, bound_hi=MVG_PRM.y_hi))
env["margin_lo"] = env.observed_min - env.bound_lo; env["margin_hi"] = env.bound_hi - env.observed_max
env.to_csv(os.path.join(RESULTS_DIR, "r1c8_plausibility_envelope.csv"), index=False); print(env.round(4))

  channel  bound_lo  observed_min  observed_max  bound_hi  margin_lo  margin_hi
0      V1       0.3        0.8759        1.0700       1.7     0.5759     0.6300
1      V2       0.3        0.5085        1.0735       1.7     0.2085     0.6265
2     Vdc       0.3        0.7166        1.2767       1.7     0.4166     0.4233
3      I1      -0.2       -0.0659        0.6566       3.0     0.1341     2.3434
4      I2      -0.2       -0.0784        0.6793       3.0     0.1216     2.3207
5   P_POI      -5.0       -1.7263        0.6406       5.0     3.2737     4.3594
6   Q_POI      -5.0       -0.4060        0.4526       5.0     4.5940     4.5474


## 4. Frozen-channel safeguards (ablation on S7, S8 - zero current after the VSC2 outage)

Both safeguards on is the implemented gate; the others are diagnostic variants only.

In [5]:
abl = []
for label, kw in [("both safeguards (implemented)", {}), ("no zero-current exemption", dict(zero_exempt_channels=())),
                  ("no excitation condition", dict(u_excitation_min=-1.0)),
                  ("neither", dict(zero_exempt_channels=(), u_excitation_min=-1.0))]:
    for sname in ["S7", "S8"]:
        lg = run_mvg(sname, SEED_REP, crip, gate_prm=dataclasses.replace(MVG_PRM, **kw))
        r = [x for x in lg["mvg_reasons"] if x]
        abl.append(dict(variant=label, scenario=sname, invalid_samples=lg["_gate"].n_invalid,
                        first_invalid_t=(float(lg["t"][np.argmax(lg["mvg_valid"] == 0)]) if r else np.nan),
                        reasons=";".join(sorted(set(r))), A_c=float(np.mean(lg["s"]))))
abl = pd.DataFrame(abl); abl.to_csv(os.path.join(RESULTS_DIR, "r1c8_frozen_safeguard_ablation.csv"), index=False); print(abl)

                         variant scenario  invalid_samples  first_invalid_t                                            reasons       A_c
0  both safeguards (implemented)       S7                0              NaN                                                     0.450775
1  both safeguards (implemented)       S8                0              NaN                                                     0.007597
2      no zero-current exemption       S7                0              NaN                                                     0.450775
3      no zero-current exemption       S8                0              NaN                                                     0.007597
4        no excitation condition       S7              792            23.98  frozen:V1,V2;frozen:V1,V2,Vdc;frozen:V1,V2,Vdc...  0.450775
5        no excitation condition       S8                0              NaN                                                     0.007597
6                        neither       S7

## 5. Focused fault-injection tests

Fault onset $t_0=15$ s on S1 (healthy, candidate active; no hazard), seed 1, plus noisy S6 (1 % noise from 10 s) for
dropout and frozen. Each case is run without the gate (reference behaviour, exceptions caught only to document them)
and with the gate. Detection delay = first invalid sample - onset; reactivation = first $s_k=1$ after the fault ends.

In [6]:
T0 = 15.0
FAULTS = {
    "F1 single NaN (Vdc)":            ("S1", [Fault("nan", T0, T0 + 0.02, 2)], True, {}),
    "F2 dropout 0.5 s (all NaN)":     ("S1", [Fault("dropout", T0, T0 + 0.5)], True, {}),
    "F3 stale 0.5 s (timestamps)":    ("S1", [Fault("stale", T0, T0 + 0.5)], True, {}),
    "F4 stale 0.5 s (no timestamps)": ("S1", [Fault("stale", T0, T0 + 0.5)], False, dict(tau_max=None)),
    "F5 3-sample delay (persistent)": ("S1", [Fault("delay", T0, 1e9, delay_steps=3)], True, {}),
    "F6 frozen V1 for 10 s":          ("S1", [Fault("frozen", T0, T0 + 10, 0)], True, {}),
    "F7 stuck-at-zero I1 for 10 s":   ("S1", [Fault("stuck_value", T0, T0 + 10, 3, 0.0)], True, {}),
    "F8 gross outlier Vdc=5 pu":      ("S1", [Fault("stuck_value", T0, T0 + 0.02, 2, 5.0)], True, {}),
    "F9 moderate outlier V1+0.2 pu":  ("S1", [Fault("offset", T0, T0 + 0.02, 0, 0.2)], True, {}),
    "F2n dropout 0.5 s, noisy S6":    ("S6", [Fault("dropout", T0, T0 + 0.5)], True, {}),
    "F6n frozen V1 10 s, noisy S6":   ("S6", [Fault("frozen", T0, T0 + 10, 0)], True, {}),
}
def fault_metrics(lg, f, gate_on):
    t, s = lg["t"], lg["s"]; t1 = min(f.t1, t[-1] + 1e-9) if len(t) else f.t1
    win = (t >= f.t0) & (t < t1)
    out = dict(exception=lg["exception"] or "", completed=len(t) == int(np.floor(lg["_sc"].T_end / pp.Ts)) + 1,
               s1_samples_in_fault=int(np.sum(s[win])), fault_samples=int(np.sum(win)),
               s1_samples_after_onset_before_t1_plus_1s=int(np.sum(s[(t >= f.t0) & (t < t1 + 1.0)])))
    if gate_on:
        inv = np.where(lg["mvg_valid"] == 0)[0]
        out.update(detected=len(inv) > 0, detect_delay_s=(float(t[inv[0]] - f.t0) if len(inv) else np.nan),
                   invalid_samples=len(inv), reasons=";".join(sorted(set(r.split(":")[0] for r in lg["mvg_reasons"] if r))),
                   dt_flag_pct_in_fault=100 * float(np.mean(lg["dt_flag"][win])) if win.any() else np.nan,
                   cri_resets=int(np.nansum(lg["mvg_reset"])),
                   nonfinite_reaching_cri=int(np.sum(~np.isfinite(lg["CRI"][lg["mvg_valid"] == 1]))))
    after = np.where((t >= t1) & (s == 1))[0]
    out["reactivation_after_fault_end_s"] = float(t[after[0]] - t1) if (len(after) and t1 < t[-1]) else np.nan
    out["n_violation_true"] = int(dcc.hard_limit_violation(lg, cons).sum()) if len(t) else np.nan
    return out
frows = []
for name, (sname, fl, ts, kw) in FAULTS.items():
    for gate_on in [False, True]:
        lg = run_mvg(sname, SEED_REP, crip, gate_prm=dataclasses.replace(MVG_PRM, **kw), use_gate=gate_on,
                     faults=fl, timestamps=ts, catch=not gate_on)
        r = fault_metrics(lg, fl[0], gate_on); r.update(case=name, scenario=sname, gate="MVG" if gate_on else "none"); frows.append(r)
ft = pd.DataFrame(frows)
ft.to_csv(os.path.join(RESULTS_DIR, "r1c8_fault_injection.csv"), index=False)
cols = ["case", "gate", "exception", "detected", "detect_delay_s", "reasons", "s1_samples_in_fault", "fault_samples",
        "reactivation_after_fault_end_s", "nonfinite_reaching_cri", "n_violation_true"]
print(ft[cols].to_string(index=False))
assert (ft[ft.gate == "MVG"].exception == "").all(), "exception with MVG enabled"
print(f"({time.time()-T_START:.0f} s)")

                          case gate                                        exception detected  detect_delay_s     reasons  s1_samples_in_fault  fault_samples  reactivation_after_fault_end_s  nonfinite_reaching_cri  n_violation_true
           F1 single NaN (Vdc) none k=750 t=15.00s LinAlgError: SVD did not converge      NaN             NaN         NaN                    0              0                             NaN                     NaN                 0
           F1 single NaN (Vdc)  MVG                                                      True            0.00   nonfinite                    0              1                            0.78                     0.0                 0
    F2 dropout 0.5 s (all NaN) none k=750 t=15.00s LinAlgError: SVD did not converge      NaN             NaN         NaN                    0              0                             NaN                     NaN                 0
    F2 dropout 0.5 s (all NaN)  MVG                                     

## 6. Recovery length and cold-start behaviour

(a) Recovery length $R\in\{5,20,40\}$ on F2 and F6 (S1): reactivation delay and the indicator values at the moment
authority returns ($\mathrm{AI}_k$ is set to 1 until $2N_a$ samples are available, so $R<2N_a$ re-activates on a partly
default-valued indicator). (b) Applying the same requirement at cold start (`startup_requires_recovery=True`) on S1-S8,
seed 1 - this would change the reference results and is reported only as an evaluated alternative.

In [7]:
rec = []
for R in [5, 20, 40]:
    for name in ["F2 dropout 0.5 s (all NaN)", "F6 frozen V1 for 10 s"]:
        sname, fl, ts, kw = FAULTS[name]
        lg = run_mvg(sname, SEED_REP, crip, gate_prm=dataclasses.replace(MVG_PRM, recovery_samples=R, **kw), faults=fl, timestamps=ts)
        t, s = lg["t"], lg["s"]; i = np.where((t >= fl[0].t1) & (s == 1))[0]
        rec.append(dict(R=R, case=name, reactivation_after_fault_end_s=(float(t[i[0]] - fl[0].t1) if len(i) else np.nan),
                        AI_at_reactivation=(float(lg["AI"][i[0]]) if len(i) else np.nan),
                        a_raw_at_reactivation=(float(lg["a_raw"][i[0]]) if len(i) else np.nan),
                        CRI_at_reactivation=(float(lg["CRI"][i[0]]) if len(i) else np.nan)))
rec = pd.DataFrame(rec); rec.to_csv(os.path.join(RESULTS_DIR, "r1c8_recovery_length.csv"), index=False); print(rec.round(4))

st = []
for sname in SCENARIO_ORDER:
    a = run_ref(sname, SEED_REP, crip); a["_sc"] = B[sname]()
    c = run_mvg(sname, SEED_REP, crip, gate_prm=dataclasses.replace(MVG_PRM, startup_requires_recovery=True))
    pa, pc = perf(a), perf(c)
    ta = a["t"][np.argmax(a["s"] == 1)] if a["s"].any() else np.nan; tc = c["t"][np.argmax(c["s"] == 1)] if c["s"].any() else np.nan
    st.append(dict(scenario=sname, first_activation_s_published=ta, first_activation_s_coldstart_R=tc,
                   **{f"{k}_published": pa[k] for k in ["N_FA", "A_c", "N_FA_H", "n_violation_true"]},
                   **{f"{k}_coldstart_R": pc[k] for k in ["N_FA", "A_c", "N_FA_H", "n_violation_true"]}))
st = pd.DataFrame(st); st.to_csv(os.path.join(RESULTS_DIR, "r1c8_coldstart_variant.csv"), index=False); print(st.round(4).to_string(index=False))

    R                        case  reactivation_after_fault_end_s  AI_at_reactivation  a_raw_at_reactivation  CRI_at_reactivation
0   5  F2 dropout 0.5 s (all NaN)                            0.64              1.0000                 0.0000               0.6432
1   5       F6 frozen V1 for 10 s                            0.08              1.0000                 0.0000               0.7045
2  20  F2 dropout 0.5 s (all NaN)                            0.64              1.0000                 0.0000               0.6432
3  20       F6 frozen V1 for 10 s                            0.38              1.0000                 0.0000               0.7056
4  40  F2 dropout 0.5 s (all NaN)                            0.78              0.8156                 0.0245               0.6606
5  40       F6 frozen V1 for 10 s                            0.78              0.9947                 0.0006               0.7646
scenario  first_activation_s_published  first_activation_s_coldstart_R  N_FA_published  A_

In [8]:
meta["bit_identical_fault_free"] = BIT_IDENTICAL
meta["runtime_s"] = round(time.time() - T_START, 1)
meta["outputs"] = sorted(f for f in os.listdir(RESULTS_DIR) if f.startswith("r1c8_"))
json.dump(meta, open(os.path.join(RESULTS_DIR, "r1c8_run_metadata.json"), "w"), indent=1, default=str)
print(f"Done in {meta['runtime_s']} s. Outputs: {meta['outputs']}")

Done in 323.9 s. Outputs: ['r1c8_baseline_unchanged_code.csv', 'r1c8_coldstart_variant.csv', 'r1c8_equivalence_fault_free.csv', 'r1c8_fault_injection.csv', 'r1c8_frozen_safeguard_ablation.csv', 'r1c8_plausibility_envelope.csv', 'r1c8_recovery_length.csv', 'r1c8_run_metadata.json']


## 7. Findings

All values below are read from the CSV files written above.

* **Fault-free equivalence.** In all 62 fault-free runs (EC-CRI 46, DC-CRI 16) the MVG runner is bit-identical to the
  reference implementation on every logged field, with the gate off and with the gate on, and the gate never
  triggers. Adding the gate therefore leaves all existing EC-CRI and DC-CRI results unchanged.
* **Plausibility envelope.** The smallest margin between the observed fault-free channel ranges and the envelope is
  0.12 pu ($I_2$, lower bound), so the envelope rejects gross outliers only.
* **Frozen-channel safeguards.** Without the channel-specific excitation condition the frozen test would falsely
  invalidate 792 samples in S7 (VSC2 outage); with both safeguards (implemented gate) there are no false triggers in S7/S8.
* **Detected faults.** Non-finite values/dropouts, stale packets (with timestamps), persistent 3-sample delay and a
  gross outlier are detected on the first faulty sample; a frozen channel and stale data without timestamps are detected
  after 0.18 s (the frozen-window length). No non-finite value reaches the CRI. Without the gate, a single NaN or a
  dropout stops the simulation with a numerical error (`LinAlgError`) in the sensitivity estimator; with the gate
  every fault case runs to completion.
* **Undetected by design.** A stuck-at-zero current sensor (zero currents are exempt because a de-energized feeder is
  legitimately constant) and a moderate +0.2 pu outlier inside the plausibility envelope are not detected.
* **Fallback is not a safety guarantee.** During 0.5 s dropout or stale-data faults on S1, the fallback running on
  held data produces 12 hard-limit violation samples (2 for stale data without timestamps). Without the gate, stale
  data leave the candidate running with no violations, and a dropout stops the simulation. Requesting the fallback prevents the candidate from acting on invalid data; it does not by itself keep
  the plant admissible.
* **Recovery.** With the default $R=\max(N,2N_a)=40$ valid samples, authority returns 0.78 s after the fault ends, with
  every CRI data window refilled by post-fault data. Shorter $R$ (5, 20) reactivates earlier while $\mathrm{AI}_k$ is
  still at its default value. Applying the same requirement at cold start delays the first activation from 0.06 s to
  0.78 s and is not used by default.
